# GDG UAM ML Contest 2026 — Clean-clone verification

This notebook checks a fresh clone of the public submission with the official harness on the 400-sample public demo. It does not train the model.

In [6]:
from pathlib import Path
import subprocess
import sys

SUBMISSION = Path("/content/gdg-uam-submission-check")
MATERIALS = Path("/content/ml-contest-materials")

if SUBMISSION.exists() or MATERIALS.exists():
    raise RuntimeError("Use a fresh Colab runtime for this clean-clone check.")

subprocess.run(
    ["git", "clone", "--depth", "1",
     "https://github.com/DavidTorro/gdg-uam-ml-contest-2026.git",
     str(SUBMISSION)],
    check=True,
)
subprocess.run(
    ["git", "clone", "--depth", "1",
     "https://github.com/GDG-UAM/ml-contest-materials.git",
     str(MATERIALS)],
    check=True,
)

revision = subprocess.check_output(
    ["git", "-C", str(SUBMISSION), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()
weight = SUBMISSION / "models" / "exp033.sesgo.onnx"
assert weight.is_file() and weight.stat().st_size > 100_000

print(f"Submission revision: {revision}")
print(f"ONNX weight: {weight.stat().st_size:,} bytes")

Submission revision: 3bbc56d
ONNX weight: 225,795 bytes


## 2. Install inference dependencies

Install the pinned packages from the submission and check that the inference libraries import successfully.

In [7]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r",
     str(SUBMISSION / "requirements.txt")],
    check=True,
)

subprocess.run(
    [sys.executable, "-c",
     'import numpy, numba, onnxruntime; '
     'print("NumPy", numpy.__version__, '
     '"| Numba", numba.__version__, '
     '"| ONNX Runtime", onnxruntime.__version__)'],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-c', 'import numpy, numba, onnxruntime; print("NumPy", numpy.__version__, "| Numba", numba.__version__, "| ONNX Runtime", onnxruntime.__version__)'], returncode=0)

## 3. Download and verify the public demo

Download the official 400-sample demo and verify its SHA-256 checksum.

In [8]:
import hashlib

versions = subprocess.check_output(
    [sys.executable, "-c",
     'import numpy, numba, onnxruntime; '
     'print("NumPy", numpy.__version__, '
     '"| Numba", numba.__version__, '
     '"| ONNX Runtime", onnxruntime.__version__)'],
    text=True,
)
print(versions.strip())

DEMO = MATERIALS / "data" / "test_demo.npz"
DEMO.parent.mkdir(parents=True, exist_ok=True)

subprocess.run(
    ["curl", "-fL", "--retry", "3", "-o", str(DEMO),
     "https://github.com/GDG-UAM/ml-contest-materials/releases/latest/download/test_demo.npz"],
    check=True,
)

digest = hashlib.sha256(DEMO.read_bytes()).hexdigest()
expected = "03fce51b3dacfaaa04418c4907def59874c4b6963c0561c9ccac83bbca90aac0"
assert digest == expected, f"Unexpected demo checksum: {digest}"

print(f"Demo verified: {DEMO.stat().st_size:,} bytes")
print(f"SHA-256: {digest}")

NumPy 2.5.4 | Numba 0.68.1 | ONNX Runtime 1.21.1
Demo verified: 1,654,340 bytes
SHA-256: 03fce51b3dacfaaa04418c4907def59874c4b6963c0561c9ccac83bbca90aac0


## 4. Run the official harness

Evaluate the cloned submission on the 400 distinct public demo samples. The harness handles setup and warmup.

In [9]:
from pathlib import Path
import subprocess
import sys

SUBMISSION = Path("/content/gdg-uam-submission-check")
MATERIALS = Path("/content/ml-contest-materials")

if not (SUBMISSION / "model.py").is_file():
    raise RuntimeError("Missing submission clone: run the clone cell first.")
if not (MATERIALS / "harness.py").is_file():
    raise RuntimeError("Missing official harness: run the clone cell first.")
if not (MATERIALS / "data" / "test_demo.npz").is_file():
    raise RuntimeError("Missing public demo: run the download cell first.")

subprocess.run(
    [sys.executable, "harness.py", "evaluate",
     "--model", str(SUBMISSION / "model.py"),
     "--source", "file:data/test_demo.npz",
     "--runs", "1"],
    cwd=MATERIALS,
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', 'harness.py', 'evaluate', '--model', '/content/gdg-uam-submission-check/model.py', '--source', 'file:data/test_demo.npz', '--runs', '1'], returncode=0)

## 5. Verify the result

Read the official harness report and check the expected public-demo result.

In [10]:
from pathlib import Path
import json
import subprocess
from IPython.display import Markdown, display

materials = Path("/content/ml-contest-materials")
submission = Path("/content/gdg-uam-submission-check")
report_path = materials / "results" / "model" / "run_0.json"
report = json.loads(report_path.read_text())

revision = subprocess.check_output(
    ["git", "-C", str(submission), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()

scored = report["n_scored"]
correct = report["n_correct"]
accuracy = 100 * report["accuracy_all"]
score = report["score"]
latency_ms = 1000 * report["latency_s"]["mean"]

passed = (
    report["status"] == "ok"
    and scored == 400
    and correct == 338
    and abs(score - 3.7789064859078927) < 0.001
)

display(Markdown(f"""
### {'✅ PASS' if passed else '❌ CHECK RESULT'} · public demo

| Check | Result |
| --- | ---: |
| Submitted revision | `{revision}` |
| Samples scored | {scored} |
| Correct predictions | {correct} |
| Accuracy | {accuracy:.1f}% |
| Demo score | {score:.6f} |
| Mean `predict` latency | {latency_ms:.4f} ms |

Detailed report: `{report_path}`
"""))

if not passed:
    raise AssertionError("The demo result differs from the expected 338/400.")


### ✅ PASS · public demo

| Check | Result |
| --- | ---: |
| Submitted revision | `3bbc56d` |
| Samples scored | 400 |
| Correct predictions | 338 |
| Accuracy | 84.5% |
| Demo score | 3.778906 |
| Mean `predict` latency | 0.1514 ms |

Detailed report: `/content/ml-contest-materials/results/model/run_0.json`
